<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_1_Foundations_Python_and_Math/Month_02_Data_Analysis_with_Pandas_and_NumPy/Week_3_Pandas_Basics/Day_21_Week_3_Review_and_Pandas_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 21: Week 3 Review and Pandas Mini-Project

## Phase 1 | Month 2 | Week 3 | Day 21

**Goal:** Consolidate all Week 3 Pandas skills and build a complete exploratory data analysis project.

### Week 3 Recap
| Day | Topic | Key Skill |
|-----|-------|-----------|
| 15 | Series | Labelled arrays, string ops |
| 16 | DataFrames | Creating, reading, inspecting |
| 17 | Indexing | .loc, .iloc, query() |
| 18 | Filtering | isin(), between(), apply() |
| 19 | Missing Data | Detect, drop, impute |
| 20 | GroupBy | Aggregation, transform, pivot |

### Resources
- 🔢 [Kaggle Pandas Exercises](https://www.kaggle.com/learn/pandas)
- 📖 [Real Python Pandas Tutorials](https://realpython.com/pandas-python-explore-dataset/)
- 🎥 [Pandas EDA Tutorial — Ken Jee](https://www.youtube.com/watch?v=8JhOO6gZU3A)

In [ ]:
import pandas as pd
import numpy as np
print('=== Week 3 Pandas Quick Reference ===')

# Create
df = pd.DataFrame({'a':[1,2,3],'b':[4,5,6],'c':['x','y','z']})
print(df.dtypes.to_dict())

# Select
print(df['a'].tolist())
print(df.loc[0:1, ['a','b']].values)

# Filter
print(df[df['a']>1][['a','c']].values)

# Impute
df2 = pd.DataFrame({'x':[1.0, None, 3.0, None, 5.0]})
print('filled:', df2['x'].fillna(df2['x'].mean()).tolist())

# GroupBy
df3 = pd.DataFrame({'g':['A','A','B','B'],'v':[10,20,30,40]})
print('group mean:', df3.groupby('g')['v'].mean().to_dict())
print('Week 3 fundamentals ready!')

## Major Project: Kenya National Exam Performance EDA

**Scenario:** You have been hired by the Ministry of Education to analyse Kenya Certificate of Secondary Education (KCSE) results across all 47 counties. Produce a comprehensive exploratory data analysis report.

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(2024)

# Generate synthetic KCSE results
n = 10_000

county_profiles = {
    'Nairobi':{'base':58,'std':12,'private_pct':0.45},
    'Kisumu': {'base':53,'std':13,'private_pct':0.25},
    'Mombasa':{'base':55,'std':12,'private_pct':0.35},
    'Nakuru': {'base':54,'std':11,'private_pct':0.30},
    'Eldoret':{'base':52,'std':13,'private_pct':0.20},
    'Kisii':  {'base':56,'std':11,'private_pct':0.15},
    'Thika':  {'base':57,'std':12,'private_pct':0.40},
    'Meru':   {'base':51,'std':14,'private_pct':0.18},
}

counties_list = list(county_profiles.keys())
county_col = np.random.choice(counties_list, n)

rows = []
for county in county_col:
    p = county_profiles[county]
    school_type = 'Private' if np.random.rand() < p['private_pct'] else 'Public'
    gender = np.random.choice(['M','F'])
    gender_bonus = 1.5 if gender == 'F' else 0
    school_bonus = 8 if school_type == 'Private' else 0
    score = np.random.normal(p['base'] + school_bonus + gender_bonus, p['std'])
    score = np.clip(score, 0, 84)
    rows.append({
        'county': county, 'school_type': school_type,
        'gender': gender, 'mean_grade': round(score, 1)
    })

df = pd.DataFrame(rows)
df['grade'] = pd.cut(df['mean_grade'],
                     bins=[0,30,40,50,60,70,80,84],
                     labels=['E','D','D+','C','C+','B','A'])

print('KCSE Dataset:')
print(df.shape)
print(df.head())
print('\nGrade distribution:')
print(df['grade'].value_counts().sort_index())

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(2024)

# Rebuild dataset
county_profiles = {'Nairobi':(58,12,0.45),'Kisumu':(53,13,0.25),'Mombasa':(55,12,0.35),
                   'Nakuru':(54,11,0.30),'Eldoret':(52,13,0.20),
                   'Kisii':(56,11,0.15),'Thika':(57,12,0.40),'Meru':(51,14,0.18)}
county_list = list(county_profiles.keys())
n = 10_000
county_col = np.random.choice(county_list, n)
rows = []
for c in county_col:
    base,std,pp = county_profiles[c]
    stype = 'Private' if np.random.rand() < pp else 'Public'
    gen   = np.random.choice(['M','F'])
    score = np.clip(np.random.normal(base + (8 if stype=='Private' else 0) + (1.5 if gen=='F' else 0), std), 0, 84)
    rows.append({'county':c,'school_type':stype,'gender':gen,'score':round(score,1)})
df = pd.DataFrame(rows)
df['grade'] = pd.cut(df['score'], bins=[0,30,40,50,60,70,80,84], labels=['E','D','D+','C','C+','B','A'])

# ==== EDA ====
print('=== 1. Overall Statistics ===')
print(df['score'].describe().round(2))

print('\n=== 2. Performance by County (sorted) ===')
county_perf = df.groupby('county')['score'].agg(['mean','std','count']).round(2)
print(county_perf.sort_values('mean', ascending=False))

print('\n=== 3. Gender Gap by County ===')
gender_gap = df.groupby(['county','gender'])['score'].mean().unstack().round(2)
gender_gap['gap'] = (gender_gap['F'] - gender_gap['M']).round(2)
print(gender_gap)

print('\n=== 4. Private vs Public ===')
print(df.groupby('school_type')['score'].describe().round(2))

print('\n=== 5. Grade A Rate per County ===')
df['is_A'] = df['grade'] == 'A'
print(df.groupby('county')['is_A'].mean().sort_values(ascending=False).map(lambda x: f'{x*100:.1f}%'))

## Challenges

1. **Advanced:** Add injection of missing data (10% in score), clean it, and re-run the analysis
2. **Visualisation:** Import `matplotlib.pyplot` and create bar charts for county mean scores
3. **Export:** Save the county summary to CSV and the full DataFrame to parquet
4. **Kaggle:** Complete all 6 Pandas exercises on Kaggle Learn (free)

In [ ]:
import pandas as pd
import numpy as np
# YOUR CHALLENGE CODE HERE

## Week 3 Complete!

### Self-Assessment
- [ ] I can create a Series and DataFrame from any source
- [ ] I use .loc/.iloc confidently without confusing them
- [ ] I handle missing data systematically
- [ ] I write groupby aggregations with multiple statistics
- [ ] I completed the KCSE EDA project

### Week 4 Preview: Data Manipulation and Merging
- Merging and joining DataFrames (like SQL JOINs)
- Reshaping: pivot, melt, stack, unstack
- Apply, map, and transform functions
- Time series with Pandas
- Month 2 final project